# 23. 질문에 맞는 근거를 찾는지 확인하기 — M6 준비

먼저 질문 목록과 **한 문제의 검색 결과**부터 확인합니다. 첫 번째 코드 셀부터 차례대로 실행하세요.

- **Hit@5:** 질문 30개 중 상위 5개 결과에 기대한 근거가 있는 비율입니다.
- **MRR@5:** 기대한 근거가 앞에 나올수록 높은 점수입니다. 1위는 1점, 2위는 0.5점이며 상위 5개에 없으면 0점입니다. 30개 질문의 점수를 평균 냅니다.

[프로젝트 적용] 기존 11·12번의 30문항과 전체 DB 검색을 재사용합니다.
[프로젝트 추가] 기대 원문 ID는 검색 순위에 넣지 않고 검색 후 비교에만 씁니다. PDF 페이지 번호만 같고 주제가 다른 결과를 정답으로 세지 않습니다.

이미 개선에 사용한 질문이므로 독립적인 새 질문 평가가 아닙니다. 그림을 이해했는지, 발췌 답변이 정확한지, 질문의 모든 조건을 충족했는지는 따로 확인해야 합니다. 새 모델이나 라이브러리를 추가하지 않습니다.


In [1]:
from pathlib import Path
import json
import sys

# notebooks에서 실행하더라도 프로젝트 루트를 찾도록 경로를 맞춥니다.
project = Path.cwd().resolve()
if project.name == "notebooks":
    project = project.parent
if not (project / "src/car_search_rag/zzong_santafe_lag").is_dir():
    raise FileNotFoundError("프로젝트 폴더 또는 notebooks 폴더에서 실행하세요.")
if str(project / "src") not in sys.path:
    sys.path.insert(0, str(project / "src"))
print("프로젝트 위치:", project)


프로젝트 위치: C:\mle-02-p1-team1


In [2]:
from car_search_rag.zzong_santafe_lag.db_evaluation_service import DbRetrievalEvaluationService

# 완료 기록에서 개인 전체 저장 작업 번호를 읽습니다. DB 비밀번호는 출력하지 않습니다.
report_path = project / "data/zzong_santafe_lag/reports/full_store_20261002.json"
store_report = json.loads(report_path.read_text(encoding="utf-8"))
if store_report["phase"] != "completed":
    raise ValueError("전체 저장 완료 기록을 먼저 확인하세요.")
run_id = store_report["save_result"]["run_id"]
# 기본값 purpose는 현재 사용 중인 글자·의미·질문 목적 결합 검색입니다.
evaluator = DbRetrievalEvaluationService(run_id, method="purpose")
print("평가 도구 준비 완료. 아직 DB 조회나 모델 실행은 하지 않았습니다.")


평가 도구 준비 완료. 아직 DB 조회나 모델 실행은 하지 않았습니다.


c:\mle-02-p1-team1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# [프로젝트 추가] DB 원문을 조회해 질문과 기대 페이지를 연결합니다. 모델은 실행하지 않습니다.
cases = evaluator.prepare_cases()
print("평가 질문 수:", len(cases))
# 처음에는 세 문제만 읽습니다. 전체 목록은 cases[:3]을 cases로 바꾸면 보입니다.
for case in cases[:3]:
    print("\n", case["case_id"], case["question"])
    for source in case["expected_sources"]:
        print("기대 PDF 쪽:", source["pdf_pages"], "| 제목:", source["title"])
print("질문 목록 식별값:", evaluator.dataset_sha256)


평가 질문 수: 30

 Q01 실외 미러를 접는 버튼은 차량 내부 안내도의 어디에 있나요?
기대 PDF 쪽: [33, 34] | 제목: 차량 내부 I

 Q02 후드 열림 레버의 위치를 보여주는 안내도를 찾아줘.
기대 PDF 쪽: [33, 34] | 제목: 차량 내부 I

 Q03 지문 인증 시스템은 차량 내부 안내도에서 몇 번인가요?
기대 PDF 쪽: [33, 34] | 제목: 차량 내부 I
질문 목록 식별값: f57629974dc8e4e0092d902256144d4d4f3723ee00c0de66192c26d51b3e2fda


## 여기까지 먼저 확인하세요

예상 출력은 **평가 질문 수: 30**입니다. 처음 세 질문의 기대 출처는 차량 내부 I 안내도, PDF 33쪽입니다. 여기까지는 질문 목록을 준비한 것이므로 아직 점수는 없습니다.

목록이 맞으면 아래에서 한 문제만 검색합니다. 첫 검색에는 기존 로컬 모델을 준비하는 시간이 걸립니다. 원본 PDF에서 실외 미러 접이 버튼 항목도 함께 대조하세요.


In [4]:
# 한 문제만 실행합니다. 전체 문서 임베딩은 다시 계산하지 않고 질문만 계산합니다.
one_result = evaluator.evaluate_case("Q01", top_k=5, progress=print)
print("\n질문:", one_result["question"])
print("기대 근거를 찾은 첫 순위:", one_result["first_relevant_rank"] or "상위 5개에 없음")
for candidate in one_result["candidates"]:
    print(candidate["rank"], candidate["title"], "| PDF 쪽:", candidate["pdf_pages"],
          "| 기대 출처 일치:", bool(candidate["matched_expected_ids"]))


DB 검색 범위 확인 완료 → 저장 당시의 로컬 모델 준비


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 571.33it/s]


질문 임베딩 완료 → 저장된 청크에서 검색·원문 연결

질문: 실외 미러를 접는 버튼은 차량 내부 안내도의 어디에 있나요?
기대 근거를 찾은 첫 순위: 1
1 차량 내부 I | PDF 쪽: [33, 34] | 기대 출처 일치: True
2 차량 내부 II | PDF 쪽: [34, 35] | 기대 출처 일치: False
3 차량 내부 III | PDF 쪽: [35, 36] | 기대 출처 일치: False
4 차량 외부 I | PDF 쪽: [30] | 기대 출처 일치: False
5 실외 미러 | PDF 쪽: [246, 247, 248, 249] | 기대 출처 일치: False


## 한 문제 확인 후 전체 30문항 실행

아래의 `run_all`을 `True`로 바꾸면 30문항을 실행합니다. 현재 검색은 질문마다 DB 자료를 읽어서 몇 분 걸릴 수 있습니다. `1/30`, `2/30`처럼 진행 상황이 나옵니다. 오류가 나면 중단하며 실패한 질문을 제외한 높은 점수를 보고하지 않습니다.

DB 자료·그림·모델 설정을 수정하지 않습니다. 결과는 메모리에 반환하며 파일로 자동 저장하지 않습니다.


In [6]:
# [프로젝트 추가] 한 문제의 결과를 확인한 다음 True로 바꿔 전체 평가를 시작합니다.
run_all = True
evaluation_report = None
if run_all:
    evaluation_report = evaluator.evaluate_all(top_k=5, progress=print)
    print("\n평가 질문 수:", evaluation_report["question_count"])
    print("Hit@5:", round(evaluation_report["metrics"]["Hit@5"], 3))
    print("MRR@5:", round(evaluation_report["metrics"]["MRR@5"], 3))
else:
    print("아직 전체 평가 전입니다. 한 문제를 확인한 뒤 run_all을 True로 바꾸세요.")


1/30 | Q01 | 실외 미러를 접는 버튼은 차량 내부 안내도의 어디에 있나요?
  기대 출처 첫 순위: 1
2/30 | Q02 | 후드 열림 레버의 위치를 보여주는 안내도를 찾아줘.
  기대 출처 첫 순위: 1
3/30 | Q03 | 지문 인증 시스템은 차량 내부 안내도에서 몇 번인가요?
  기대 출처 첫 순위: 1
4/30 | Q04 | 유리창 잠금 버튼과 전자식 차일드 락 버튼 위치를 알려줘.
  기대 출처 첫 순위: 1
5/30 | Q05 | 시동 버튼 위치를 보여주는 안내도가 필요해.
  기대 출처 첫 순위: 1
6/30 | Q06 | 스마트폰 무선 충전 표시등은 내부 안내도의 어느 항목인가요?
  기대 출처 첫 순위: 1
7/30 | Q07 | 글로브 박스와 동승석 트레이의 위치를 찾아줘.
  기대 출처 첫 순위: 1
8/30 | Q08 | 주차 안전 버튼과 주차 뷰 버튼 위치가 나오는 안내도를 보여줘.
  기대 출처 첫 순위: 1
9/30 | Q09 | 와이퍼와 워셔 스위치는 내부 안내도 몇 번인가요?
  기대 출처 첫 순위: 1
10/30 | Q10 | 블루투스 핸즈프리 버튼과 음성 인식 버튼은 어디에 있나요?
  기대 출처 첫 순위: 1
11/30 | Q11 | 패들 쉬프트 위치를 보여주는 내부 안내도를 찾아줘.
  기대 출처 첫 순위: 1
12/30 | Q12 | 차간거리 버튼과 차로 주행 보조 버튼 위치를 찾아줘.
  기대 출처 첫 순위: 1
13/30 | Q13 | 1.6 T-GDi HEV에 표시된 엔진 오일 SAE 점도는 무엇인가요?
  기대 출처 첫 순위: 1
14/30 | Q14 | 0W-20이 표시된 엔진 오일 점도 표를 보여줘.
  기대 출처 첫 순위: 1
15/30 | Q15 | 엔진 오일 SAE 점도 표의 온도 눈금은 어디부터 어디까지인가요?
  기대 출처 첫 순위: 1
16/30 | Q16 | 안전벨트를 착용할 때 골반띠는 어디를 지나야 하나요?
  기대 출처 첫 순위: 3
17/30 | Q17 | 안전벨트 버클을 얼마나

In [7]:
# 기대 출처를 놓친 질문을 모아 다음 개선 대상을 정합니다. 정답을 결과에 맞춰 바꾸지 않습니다.
if evaluation_report is not None:
    missed = [row for row in evaluation_report["results"] if not row["hit_at_k"]]
    print("상위 5개에서 기대 출처를 찾지 못한 질문 수:", len(missed))
    for row in missed:
        print(row["case_id"], row["question"])
        for source in row["expected_sources"]:
            print("  기대 PDF 쪽:", source["pdf_pages"], "| 제목:", source["title"])
else:
    print("전체 평가 후 확인하는 셀입니다.")


상위 5개에서 기대 출처를 찾지 못한 질문 수: 0


In [9]:
# 기대한 근거를 1위에서 찾지 못한 질문만 확인합니다.
for 결과 in evaluation_report["results"]:
    if 결과["first_relevant_rank"] != 1:
        print("\n질문:", 결과["case_id"], 결과["question"])
        print("기대한 근거의 순위:", 결과["first_relevant_rank"])

        # 앞에 나온 후보들의 제목과 PDF 페이지를 살펴봅니다.
        for 후보 in 결과["candidates"]:
            print(
                후보["rank"], "위:",
                후보["title"],
                "| PDF 쪽:", 후보["pdf_pages"],
                "| 기대 출처 일치:",
                bool(후보["matched_expected_ids"]),
            )


질문: Q16 안전벨트를 착용할 때 골반띠는 어디를 지나야 하나요?
기대한 근거의 순위: 3
1 위: 안전벨트 사용 시 유의 사항 | PDF 쪽: [113, 114] | 기대 출처 일치: False
2 위: 안전벨트 구속 장치 | PDF 쪽: [105, 106, 107, 108, 109, 110, 111, 112] | 기대 출처 일치: False
3 위: 안전벨트 착용 | PDF 쪽: [54] | 기대 출처 일치: True
4 위: 안전벨트의 적절한 관리 | PDF 쪽: [114] | 기대 출처 일치: False
5 위: 안전벨트 착용의 중요성 | PDF 쪽: [126, 127] | 기대 출처 일치: False

질문: Q17 안전벨트 버클을 얼마나 밀어 넣어야 하나요?
기대한 근거의 순위: 3
1 위: 안전벨트 구속 장치 | PDF 쪽: [105, 106, 107, 108, 109, 110, 111, 112] | 기대 출처 일치: False
2 위: 뒷좌석 | PDF 쪽: [85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100] | 기대 출처 일치: False
3 위: 안전벨트 착용 | PDF 쪽: [54] | 기대 출처 일치: True
4 위: 어린이 보호용 장치의 장착 | PDF 쪽: [115, 116, 117, 118, 119, 120] | 기대 출처 일치: False
5 위: 안전벨트 프리텐셔너 | PDF 쪽: [112, 113] | 기대 출처 일치: False

질문: Q19 차단 클립을 버클에 꽂으면 경고등과 경고음은 어떻게 되나요?
기대한 근거의 순위: 4
1 위: 경보 상태 | PDF 쪽: [227] | 기대 출처 일치: False
2 위: 안전벨트 구속 장치 | PDF 쪽: [105, 106, 107, 108, 109, 110, 111, 112] | 기대 출처 일치: False
3 위: 경고등 및 표시등 | PDF 쪽: [154, 155, 156, 157, 158, 159, 160

## 점수를 읽을 때 주의할 점

- 주 지표는 본문·필수 각주를 합친 근거 묶음 안에 기대 원문 ID가 하나 이상 있으면 성공입니다. 대표 원문만 비교한 `primary_Hit@5`·`primary_MRR@5`도 결과에 따로 남습니다.
- 여러 기대 ID는 기존 평가의 허용 출처입니다. 하나를 찾았다고 복합 질문 전체에 답할 수 있다는 뜻은 아닙니다.
- MRR@5는 상위 5개 밖의 순위를 모르는 제한 지표입니다. 전체 순위 MRR과 구분합니다.
- 기존 30문항은 안내도·점도·안전벨트·오일·패들 쉬프트 중심입니다. PDF 전체 주제와 근거 없는 질문을 대표하지 않습니다.
- 그림 설명과 파일 연결의 정확도는 PDF 원본 화면과 대조해야 합니다. 이 점수는 이미지 임베딩 평가가 아닙니다.
- M5의 발췌 판단 보완과 새 질문 평가가 남아 있습니다. M6 전체 완료나 성능 목표 달성으로 표시하지 않습니다.

반복 실행 코드는 `db_evaluation_service.py`에 있습니다. 노트북은 질문과 결과를 한 단계씩 확인하는 기록장입니다.
